# Create Virtual Machine: CSC Pouta

This notebook is a step-by-step  lab guide to explain the concepts in creating virtual machine in pouta service and includes hands-on exercises.

> Note: Some commands require access to CSC Pouta, Horizon, OpenStack CLI, Allas, and a real VM. Run those cells only in the correct environment.

## Learning Objectives

By the end of this course, students should be able to:

1. Explain IaaS, cloud, VM, OpenStack, Pouta, Rahti, and HPC systems.
2. Create and connect to a VM in Pouta.
3. Understand private/public networks, routers, floating IPs, and security groups.
4. Use persistent volumes and Allas object storage correctly.
5. Design a workflow for machine learning using VM + volume + Allas.
6. Take snapshots of VM and volumes.
7. Use OpenStack CLI for basic operations.

# Part 1 — Cloud, VM, IaaS, and OpenStack

## VM
A virtual machine is a remote Linux computer running on physical hardware in a data center.

## Cloud
Cloud means a pool of remote resources such as CPU, RAM, storage, and network that users can request on demand.

## IaaS
Infrastructure as a Service gives you low-level infrastructure:
- VM
- Storage
- Network
- IP addresses

Examples:
- CSC Pouta
- AWS EC2
- Azure Virtual Machines
- Google Compute Engine

## OpenStack
OpenStack is software that manages cloud infrastructure. It allows users to create VMs, networks, routers, volumes, and floating IPs.

## Exercise 1 — Concept Check

Answer in your own words:

1. What is the difference between a VM and a physical server?
2. Why is Pouta considered IaaS?
3. What does OpenStack manage?
4. Why is a cloud more flexible than using a single local PC?

# Part 2 — Pouta, Rahti, and HPC

## Pouta
Pouta is an OpenStack-based cloud service where you create VMs and manage them yourself. You usually have `sudo`/root-level control.

## Rahti
Rahti is a container platform based on Kubernetes/OpenShift. You deploy containerized applications but usually do not have root access.

## HPC: Puhti, Mahti, LUMI
HPC systems are for batch jobs and large-scale computations.

Typical usage:
- Puhti: general CPU and bioinformatics workflows
- Mahti: large CPU-parallel workloads
- LUMI: large GPU/HPC workloads

## Decision Table

| Need | Best choice |
|---|---|
| VM with root access | Pouta |
| Deploy containerized web app/API | Rahti |
| CPU-parallel batch job | Puhti or Mahti |
| Large GPU deep learning | LUMI |
| Persistent database/app service | Pouta or DBaaS |

## Exercise 2 — Choose the Right Platform

Choose Pouta, Rahti, Puhti/Mahti, or LUMI:

1. You want to host a FastAPI model inference API.
2. You want to train a large deep learning model on GPUs.
3. You want a VM where you can install any Linux package.
4. You want to run a CPU-parallel simulation.
5. You want to deploy a Dockerized app without managing a VM.

# Part 3 — Creating and Connecting to a Pouta VM

General workflow:

1. Create a CSC account.
2. Create a CSC project.
3. Activate cPouta service.
4. Create or import SSH key pair.
5. Create a VM.
6. Open SSH in security group.
7. Attach floating IP.
8. Connect with SSH.

## Step 1 — Create or Check SSH Key Pair

Run locally on your laptop, not inside the VM.

In [ ]:
# Check whether you already have SSH keys
!ls -la ~/.ssh

In [ ]:
# Create a new SSH key if needed
# Uncomment and run in a terminal, not necessarily inside Jupyter:
# ssh-keygen -t ed25519 -f ~/.ssh/pouta_key

## Step 2 — Launch VM in Horizon

In Horizon:

1. Go to **Compute → Instances**.
2. Click **Launch Instance**.
3. Choose:
   - Name
   - Image: Ubuntu / Rocky / AlmaLinux
   - Flavor: VM size
   - Network
   - Key pair
4. Launch the instance.

## Step 3 — Security Group for SSH

Open port 22:

- Protocol: TCP
- Port: 22
- Source: your IP or `0.0.0.0/0` for testing only

Better security: restrict SSH to your own IP.

## Step 4 — Floating IP

A VM normally has a private IP. To connect from outside, attach a floating IP.

Private IP example:
`192.168.0.5`

Public/Floating IP example:
`86.50.x.x`

## Step 5 — SSH into the VM

Use the correct username for the image:

| Image | Username |
|---|---|
| Ubuntu | ubuntu |
| Rocky Linux | rocky |
| CentOS | centos |
| Debian | debian |

In [ ]:
# Example SSH command
# Replace key path, username, and IP:
# ssh -i ~/.ssh/pouta_key rocky@PUBLIC_IP

## Exercise 3 — SSH Troubleshooting

For each case, write the likely cause:

1. `Permission denied (publickey)`
2. `Connection timed out`
3. `Too many authentication failures`
4. You used `root@PUBLIC_IP` and login failed
5. SSH worked yesterday but not today after reassigning floating IP

# Part 4 — Networking in Pouta

## Main Concepts

- **Internal/private network:** network where VMs communicate using private IPs.
- **External/public network:** network connected to the internet.
- **Router:** connects internal network to external network.
- **Floating IP:** public IP mapped to a VM.
- **Security group:** firewall rules.

## Typical Pouta Network Topology

```text
Laptop
  ↓
Internet
  ↓
Floating IP
  ↓
Virtual Router
  ↓
Private Network
  ↓
VM
```

VMs can talk to each other through private IPs, but inbound internet access requires floating IP and firewall rules.

## Exercise 4 — Design a Secure Network

Design a setup with:

- VM1 = API server, public
- VM2 = database server, private

Questions:

1. Which VM gets a floating IP?
2. Which VM should expose port 22?
3. Should the database expose port 5432 to the internet?
4. How should VM1 reach VM2?

# Part 5 — Storage in Pouta

Pouta storage types:

1. **Root disk**
   - Contains OS
   - Deleted when VM is deleted

2. **Ephemeral storage**
   - Temporary
   - Fast but unreliable
   - Not included in VM snapshot

3. **Persistent volume**
   - Separate disk
   - Survives VM deletion
   - Attach to one VM at a time

4. **Allas object storage**
   - Large object storage
   - API-based
   - Not a normal filesystem

## File Storage vs Object Storage

| Feature | Persistent Volume | Allas Object Storage |
|---|---|---|
| Looks like disk | Yes | No |
| Direct edit | Yes | No |
| Access method | OS filesystem | API / a-commands |
| Good for ML training | Yes | Not directly |
| Good for backup/archive | Yes, smaller scale | Yes, large scale |

## Important Rule

```text
Allas = warehouse
Volume = workspace
VM = compute
```

For ML:

```text
Allas → download → Volume → train → upload results → Allas
```

# Part 6 — Working with Persistent Volumes

After creating a volume in Horizon and attaching it to a VM, it appears as a new block device.

In [ ]:
# Run inside VM to inspect disks
# lsblk

In [ ]:
# Example: format and mount a new volume
# WARNING: mkfs erases data on the device. Use only on a new empty volume.
# sudo mkfs.ext4 /dev/vdb
# sudo mkdir -p /data
# sudo mount /dev/vdb /data
# df -h

## Exercise 5 — Volume Planning

You have:
- 10 GB dataset
- 20 GB intermediate files
- 5 GB model outputs

Questions:

1. What volume size would you create?
2. Why should you not store everything on root disk?
3. What should be uploaded back to Allas?

# Part 7 — Working with Allas

Allas is object storage. You cannot directly edit files inside Allas like a normal disk.

Workflow:

```text
download → modify → upload
```

In [ ]:
# Example Allas commands
# a-list
# a-get bucket-name/dataset.csv
# a-put results.csv

## Example ML Workflow with 10 GB Dataset

1. Create VM.
2. Attach 50–100 GB persistent volume.
3. Download dataset from Allas to `/data`.
4. Train model using files under `/data`.
5. Save model and results.
6. Upload model/results to Allas.

In [ ]:
# Example directory structure
# mkdir -p /data/project/{raw,processed,models,results}
# cd /data/project

In [ ]:
# Example Python loading from volume
import os
data_path = "/data/project/raw"
model_path = "/data/project/models/model.pt"

print("Data path:", data_path)
print("Model output path:", model_path)

## Exercise 6 — 4 TB Dataset Scenario

Your database is 4 TB in Allas, but your volume quota is 1 TB.

Design a strategy using one of:

1. Subset selection
2. Chunking
3. Streaming
4. Request more quota
5. Move computation to suitable HPC/storage environment

Write your proposed workflow.

# Part 8 — Machine Learning Setup on VM

Use the VM like a normal Linux server.

For Ubuntu:
```bash
sudo apt update
sudo apt install python3 python3-venv git
```

For Rocky/CentOS:
```bash
sudo dnf install python3 git
```

In [ ]:
# Example Python virtual environment commands
# python3 -m venv ~/venvs/ml
# source ~/venvs/ml/bin/activate
# pip install --upgrade pip
# pip install numpy pandas scikit-learn torch

## Simple Example Python ML Cell

This cell is executable anywhere and shows the idea of loading data and training a simple model.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))

## Exercise 7 — ML Pipeline Design

For a cell-line dataset:

1. Where do you store raw data?
2. Where do you store temporary processed data?
3. Where do you train the model?
4. Where do you save final model files?
5. What should be backed up?

# Part 9 — Snapshots

Snapshot = saved state of a VM or volume.

Important:
- VM snapshot does not include ephemeral storage.
- If data is on a volume, take a volume snapshot separately.
- Best practice: power off VM before snapshot.

## VM Snapshot via Horizon

1. Go to **Compute → Instances**.
2. Stop the VM.
3. Click **Actions → Create Snapshot**.
4. Name it clearly.
5. Find it under **Compute → Images**.

## Volume Snapshot via Horizon

1. Go to **Volumes → Volumes**.
2. Select the volume.
3. Create snapshot.
4. Find it under **Volumes → Snapshots**.

## Exercise 8 — Backup Strategy

You have:
- VM with Python environment
- Persistent volume with data
- Results uploaded to Allas

What snapshots/backups do you need before starting a risky software update?

# Part 10 — OpenStack CLI

CLI lets you manage Pouta from terminal.

Setup:

1. Install Python.
2. Install OpenStack client.
3. Download project RC file from Horizon.
4. Source the RC file.
5. Run OpenStack commands.

In [ ]:
# Install client
# pip install --user python-openstackclient

In [ ]:
# Source RC file
# source project-openrc.sh

In [ ]:
# Useful commands
# openstack server list
# openstack volume list
# openstack floating ip list
# openstack network list
# openstack security group list

## Exercise 9 — CLI Practice

Run these commands and record the output:

1. `openstack server list`
2. `openstack network list`
3. `openstack volume list`
4. `openstack floating ip list`

Then answer:
- How many VMs exist?
- How many floating IPs are allocated?
- How many volumes exist?

# Part 11 — Common SSH Problems

Checklist:

1. Is VM running?
2. Is floating IP attached?
3. Is port 22 open in security group?
4. Is username correct?
5. Is private key correct?
6. Are file permissions correct?

In [ ]:
# Fix private key permissions
# chmod 600 ~/.ssh/pouta_key

# Final Assignment — Build a Full Pouta ML Workflow

## Task

Design and document a full workflow for a 10 GB cell-line dataset stored in Allas.

Your answer must include:

1. VM flavor choice and justification.
2. Network setup.
3. Security group rules.
4. Storage design.
5. Allas download/upload workflow.
6. ML environment setup.
7. Snapshot/backup strategy.
8. What changes if the dataset grows to 4 TB?

## Deliverable

Submit:
- A diagram
- Command list
- Short explanation of each component

# Final Summary

Key concepts:

```text
Pouta = VM-based cloud
Rahti = container platform
LUMI/Puhti/Mahti = HPC systems
Allas = object storage
Volume = persistent working disk
Floating IP = public access to VM
Security group = firewall
Snapshot = saved state
```

Golden rule:

```text
Use Allas for storing large datasets.
Use volumes for active computation.
Use VMs for running code.
Use snapshots/backups before risky changes.
```